# 🤖 TravelMate AI - 06 Semantic Recommender

This notebook upgrades our text recommender from **word matching** to **semantic similarity**.

### Goal

A user can write:

> "I want quiet scenic places where I can take beautiful pictures."

The model creates embeddings for the query and each place, then ranks places by semantic similarity.

### Pipeline

```text
Place text
    ↓
Sentence Transformer
    ↓
Embedding vector
    ↓
User query embedding
    ↓
Cosine similarity
    ↓
Semantic ranking
    ↓
🎯 Recommendations
```

We use a pretrained Sentence Transformer for inference. We are not training the transformer from scratch.


## 1. Import libraries

In [1]:
import os
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully! ✅")


d:\college_work\PG\linkedIn_projects\TravelMate-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully! ✅


## 2. Load the enriched dataset

In [2]:
df = pd.read_csv("../data/processed/manali_places_enriched.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (20, 20)


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,nature,history,culture,adventure,photography,shopping,religious,family,travel_tags,interest_count
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,1,1,1,0,1,1,1,0,"culture, history, nature, photography, religio...",6
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,1,1,1,0,1,0,0,1,"culture, family, history, nature, photography",5
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,0,1,0,0,1,0,0,0,"history, photography",2
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,0,0,0,0,0,0,0,0,NaN,0
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,1,1,0,0,1,0,0,1,"family, history, nature, photography",4


## 3. Build semantic text

Our current dataset does not yet contain rich descriptions, so we combine the best text fields currently available.


In [3]:
df["semantic_text"] = (
    df["name"].fillna("") + ". "
    + df["category"].fillna("") + ". "
    + df["travel_tags"].fillna("")
)

df[["name", "semantic_text"]].head(10)


,name,semantic_text
0,Hadimba Devi Temple,Hadimba Devi Temple. Tourist attraction. cultu...
1,Old Manali snow point,Old Manali snow point. Tourist attraction. cul...
2,Nehru Kund,"Nehru Kund. Tourist attraction. history, photo..."
3,Kullu Manali River rafting,Kullu Manali River rafting. Tourist attraction.
4,Jogini Falls,"Jogini Falls. Tourist attraction. family, hist..."
5,Van Vihar National Park,Van Vihar National Park. Tourist attraction. f...
6,Manali View Point,Manali View Point. Tourist attraction. photogr...
7,Rahala Waterfalls,"Rahala Waterfalls. Tourist attraction. family,..."
8,Lama Dugh Trek Start Point,Lama Dugh Trek Start Point. Tourist attraction...
9,Atal Bihari statue,Atal Bihari statue. Tourist attraction. history


## 4. Load Sentence Transformer

We use `all-MiniLM-L6-v2`, a relatively lightweight sentence-embedding model.

The first run may download the model. Later runs can use the cached model.


In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Sentence Transformer loaded successfully! ✅")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6627.90it/s]


Sentence Transformer loaded successfully! ✅


## 5. Create embeddings for all places

Each place is converted into a dense numerical vector.


In [5]:
place_embeddings = model.encode(
    df["semantic_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", place_embeddings.shape)


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.36it/s]

Embedding matrix shape: (20, 384)


## 6. Inspect one embedding

In [6]:
print("Place:", df.loc[0, "name"])
print("Embedding dimensions:", len(place_embeddings[0]))
print("First 10 values:")
print(place_embeddings[0][:10])


Place: Hadimba Devi Temple
Embedding dimensions: 384
First 10 values:
[ 0.06891597  0.0542528  -0.07661545  0.0998638  -0.0502775   0.00354844
  0.01979782 -0.00265913 -0.08154238  0.01681263]


## 7. Build semantic recommendation function

In [7]:
def recommend_semantic(query, top_n=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    similarity_scores = cosine_similarity(
        query_embedding,
        place_embeddings
    ).flatten()

    result = df.copy()
    result["semantic_similarity"] = similarity_scores

    return result.sort_values(
        "semantic_similarity",
        ascending=False
    )[[
        "name",
        "category",
        "rating",
        "reviews",
        "travel_tags",
        "semantic_similarity"
    ]].head(top_n).reset_index(drop=True)


## 8. Test Query 1 🌿📸

In [8]:
query_1 = "I want quiet scenic places where I can take beautiful pictures"

recommend_semantic(query_1, top_n=5)


,name,category,rating,reviews,travel_tags,semantic_similarity
0,Manali View Point,Tourist attraction,4.6,87,photography,0.560242
1,Van Vihar National Park,Tourist attraction,4.2,9050,"family, history, nature, photography",0.470848
2,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.461932
3,Kharma valley,Tourist attraction,4.8,143,"history, nature",0.428529
4,Hadimba Forest Block,Tourist attraction,4.4,25,NaN,0.426823


## 9. Test Query 2 🏛️🛕

In [9]:
query_2 = "I want to explore old temples and cultural attractions"

recommend_semantic(query_2, top_n=5)


,name,category,rating,reviews,travel_tags,semantic_similarity
0,Shiv Mahadev Temple,Tourist attraction,4.6,270,"history, religious",0.640498
1,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.628577
2,Old Manali View point,Tourist attraction,4.7,44,"family, history",0.521615
3,Atal Bihari statue,Tourist attraction,4.5,74,history,0.509593
4,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.506953


## 10. Test Query 3 🥾🏔️

In [10]:
query_3 = "I want exciting outdoor activities and mountain adventures"

recommend_semantic(query_3, top_n=5)


,name,category,rating,reviews,travel_tags,semantic_similarity
0,Himalayan Igloo,Tourist attraction,4.5,199,NaN,0.404483
1,Kharma valley,Tourist attraction,4.8,143,"history, nature",0.400159
2,Lama Dugh Trek Start Point,Tourist attraction,4.6,297,nature,0.398660
3,Baror Parsha Waterfall,Tourist attraction,4.7,489,"history, nature",0.367151
4,Old Manali snow point,Tourist attraction,4.6,428,"culture, family, history, nature, photography",0.355168


## 11. Test Query 4 👨‍👩‍👧

In [11]:
query_4 = "I am travelling with my family and want enjoyable outdoor places"

recommend_semantic(query_4, top_n=5)


,name,category,rating,reviews,travel_tags,semantic_similarity
0,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.409805
1,Kullu Manali River rafting,Tourist attraction,4.5,88,NaN,0.400960
2,Van Vihar National Park,Tourist attraction,4.2,9050,"family, history, nature, photography",0.400572
3,Hadimba Forest Block,Tourist attraction,4.4,25,NaN,0.381364
4,Manali Bazaar,Tourist attraction,4.3,3991,"family, history, nature, shopping",0.376461


## 12. Compare the meaning of two queries

The following tests whether two differently worded queries are semantically related.


In [12]:
query_a = "peaceful natural photography spots"
query_b = "quiet scenic places for taking pictures"

embedding_a = model.encode(
    [query_a],
    normalize_embeddings=True
)

embedding_b = model.encode(
    [query_b],
    normalize_embeddings=True
)

query_similarity = cosine_similarity(
    embedding_a,
    embedding_b
)[0][0]

print("Semantic similarity:", round(float(query_similarity), 4))


Semantic similarity: 0.6537


## 13. Compare TF-IDF with Semantic AI

Notebook 06 has its own TF-IDF objects, so it does not depend on variables from notebook 05.

This makes the notebook self-contained.


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Prepare text for TF-IDF
df["tfidf_text"] = (
    df["name"].fillna("") + " "
    + df["category"].fillna("") + " "
    + df["travel_tags"].fillna("")
)

# TF-IDF preprocessing
def clean_text(text):
    import re
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["tfidf_text"] = df["tfidf_text"].apply(clean_text)

# Fit TF-IDF model locally in this notebook
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix_local = tfidf_vectorizer.fit_transform(
    df["tfidf_text"]
)

query = "quiet scenic places for taking pictures"

# TF-IDF recommendations
tfidf_query_vector = tfidf_vectorizer.transform(
    [clean_text(query)]
)

tfidf_scores = cosine_similarity(
    tfidf_query_vector,
    tfidf_matrix_local
).flatten()

tfidf_result = df.copy()
tfidf_result["tfidf_score"] = tfidf_scores

print("📝 TF-IDF Recommendations")

display(
    tfidf_result.sort_values(
        "tfidf_score",
        ascending=False
    )[[
        "name",
        "travel_tags",
        "tfidf_score"
    ]].head(5)
)

# Semantic recommendations
print("\n🧠 Semantic Recommendations")

display(
    recommend_semantic(
        query,
        top_n=5
    )
)


📝 TF-IDF Recommendations


,name,travel_tags,tfidf_score
0,Hadimba Devi Temple,"culture, history, nature, photography, religio...",0.0
1,Old Manali snow point,"culture, family, history, nature, photography",0.0
2,Nehru Kund,"history, photography",0.0
3,Kullu Manali River rafting,NaN,0.0
4,Jogini Falls,"family, history, nature, photography",0.0



🧠 Semantic Recommendations


,name,category,rating,reviews,travel_tags,semantic_similarity
0,Manali View Point,Tourist attraction,4.6,87,photography,0.535230
1,Van Vihar National Park,Tourist attraction,4.2,9050,"family, history, nature, photography",0.447241
2,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.418287
3,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.417762
4,Hadimba Forest Block,Tourist attraction,4.4,25,NaN,0.378226


## 14. Add rating to the semantic model

Semantic relevance is the main signal and rating is a secondary signal.

Baseline:

```text
85% → Semantic similarity
15% → Normalized rating
```

These weights are a starting point, not learned weights.


In [14]:
rating_min = df["rating"].min()
rating_max = df["rating"].max()

if rating_max == rating_min:
    df["rating_normalized"] = 1.0
else:
    df["rating_normalized"] = (
        (df["rating"] - rating_min)
        / (rating_max - rating_min)
    )


def recommend_semantic_hybrid(query, top_n=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    similarity_scores = cosine_similarity(
        query_embedding,
        place_embeddings
    ).flatten()

    result = df.copy()
    result["semantic_similarity"] = similarity_scores

    result["hybrid_score"] = (
        0.85 * result["semantic_similarity"]
        + 0.15 * result["rating_normalized"]
    )

    return result.sort_values(
        "hybrid_score",
        ascending=False
    )[[
        "name",
        "rating",
        "reviews",
        "travel_tags",
        "semantic_similarity",
        "hybrid_score"
    ]].head(top_n).reset_index(drop=True)


## 15. Test the semantic hybrid recommender

In [15]:
recommend_semantic_hybrid(
    "quiet scenic places for taking pictures",
    top_n=5
)


,name,rating,reviews,travel_tags,semantic_similarity,hybrid_score
0,Manali View Point,4.6,87,photography,0.535230,0.571612
1,Jogini Falls,4.6,10842,"family, history, nature, photography",0.418287,0.472210
2,Kharma valley,4.8,143,"history, nature",0.361882,0.457599
3,Nehru Kund,4.4,7767,"history, photography",0.417762,0.438431
4,Old Manali snow point,4.6,428,"culture, family, history, nature, photography",0.374756,0.435209


## 16. Test several natural-language queries

In [16]:
test_queries = [
    "peaceful places surrounded by nature",
    "beautiful locations for taking photos",
    "historical and spiritual places",
    "family friendly outdoor attractions",
    "adventure and trekking experiences"
]

for query in test_queries:
    print(f"\n🔎 Query: {query}")
    display(
        recommend_semantic_hybrid(query, top_n=3)
    )



🔎 Query: peaceful places surrounded by nature


,name,rating,reviews,travel_tags,semantic_similarity,hybrid_score
0,Kharma valley,4.8,143,"history, nature",0.324091,0.425478
1,Gulaba Viewpoint,4.5,3576,"family, history, nature",0.341145,0.389973
2,Himalayan Igloo,4.5,199,NaN,0.320861,0.372732



🔎 Query: beautiful locations for taking photos


,name,rating,reviews,travel_tags,semantic_similarity,hybrid_score
0,Manali View Point,4.6,87,photography,0.598127,0.625075
1,Jogini Falls,4.6,10842,"family, history, nature, photography",0.517325,0.556393
2,Kharma valley,4.8,143,"history, nature",0.422169,0.508844



🔎 Query: historical and spiritual places


,name,rating,reviews,travel_tags,semantic_similarity,hybrid_score
0,Kharma valley,4.8,143,"history, nature",0.457621,0.538978
1,Shiv Mahadev Temple,4.6,270,"history, religious",0.495070,0.537476
2,Old Manali View point,4.7,44,"family, history",0.450592,0.516336



🔎 Query: family friendly outdoor attractions


,name,rating,reviews,travel_tags,semantic_similarity,hybrid_score
0,Jogini Falls,4.6,10842,"family, history, nature, photography",0.472091,0.517944
1,Mini Switzerland Manali,4.6,79,family,0.456948,0.505072
2,Old Manali View point,4.7,44,"family, history",0.410199,0.482002



🔎 Query: adventure and trekking experiences


,name,rating,reviews,travel_tags,semantic_similarity,hybrid_score
0,Lama Dugh Trek Start Point,4.6,297,nature,0.486832,0.530474
1,Kullu Manali River rafting,4.5,88,NaN,0.349310,0.396914
2,Kharma valley,4.8,143,"history, nature",0.269812,0.379340


## 17. Save semantic artifacts

We save embeddings so they do not need to be recalculated every time.


In [17]:
embedding_path = "../data/embeddings"
os.makedirs(embedding_path, exist_ok=True)

np.save(
    f"{embedding_path}/manali_place_embeddings.npy",
    place_embeddings
)

df.to_csv(
    "../data/processed/manali_places_semantic.csv",
    index=False
)

print("✅ Embeddings saved!")
print("✅ Semantic dataset saved!")


✅ Embeddings saved!
✅ Semantic dataset saved!


# 🎯 What we achieved

We now have three recommendation generations:

### Model 1 - Structured Feature Recommender
```text
Travel features → User vector → Cosine similarity
```

### Model 2 - TF-IDF
```text
Text → TF-IDF → Cosine similarity
```

### Model 3 - Semantic AI
```text
Text → Sentence Transformer → Embedding → Cosine similarity
```

### Important limitation

Our current place text is still short:

```text
name + category + travel_tags
```

The semantic model cannot infer facts that are not present in the text. Later, we will enrich the dataset with richer descriptions, activities, duration, pricing and opening hours.

## Next: `07_hybrid_recommender.ipynb`

We will combine structured features, TF-IDF, semantic embeddings, rating and popularity into the main TravelMate ranking engine. 🔥
